- When a factory makes EV batteries, every cell comes out slightly different. 
 Most are fine, but a few are secretly defective: they look normal on day one and then degrade fast after a few hundred charge/discharge cycles.
- Right now, the only sure way to find out whether a cell is good is to cycle it 200+ times and watch what happens. 
 That takes weeks or months, costs a lot of electricity and test-rig time, and it’s impractical to do for every cell.
 If a bad cell slips into a battery pack, the whole pack can underperform or become a safety risk.

 **Use only the first few cycles of test data (days of testing, not months) to answer two questions:**
- Which cells look abnormal compared to the rest of their batch? (anomaly detection) 
- How much capacity will each cell still have at cycle 200? (prediction) 
 
 **We also make assumptions about the cycle 200 testing using readings upto cycle 100 to understand the accuracy of our method**

## WE ARE GOING TO USE ALL IMPORTS THAT ARE NEEDED AT THE START OF EACH CELL. EVEN THOUGH IMPORTS MAY REPEAT, IT WILL HELP WHEN REVISITING THE CODE

In [1]:
import pandas as pd

data = pd.read_csv("data/battery_cells_wide_dataset.csv")
print(data.columns)

FileNotFoundError: [Errno 2] No such file or directory: 'data/battery_cells_wide_dataset.csv'

In [155]:
# Adding delta X values for both cases
# WE CAN CHECK FOR DIFFERENT VALUES BUT WE GET SATISFACTORY RESULTS FOR UPTO CYCLE 25 FOR CLASSIFICATION 
V_limit = 'V_cycle25'
R_int_limit = 'R_int_cycle25'
C_limit = 'C_cycle25'
T_limit = 'T_cycle25'
data['Delta_V'] = data['V_cycle0']- data[V_limit]
data['Delta_R_int'] = data['R_int_cycle0']- data[R_int_limit]
data['Delta_C'] = data['C_cycle0']- data[C_limit]
data['Delta_T'] = data['T_cycle0']- data[T_limit]

## CLASSIFICATION PROBLEM - IDENTIFY OUTLIERS USING DIFFERENT METHODS

#### Z SCORE

In [156]:
import numpy as np

# Anomalies are more then 3 MAD away from the median of the dist.
# MAD = k * median(|x-median(x)|)

x = data[['Delta_V', 'Delta_R_int', 'Delta_C', 'Delta_T']] 
k = 1.48  # GENERALLY FOR  A NORMAL DIST. WE USE k = 1.48

median_V = data['Delta_V'].median()
MAD_V = (k) * (abs(data['Delta_V'] - median_V)).median()  # MEDIAN ABSOLUTE DEVIATION
anomaly_V = ((data['Delta_V']) > (median_V + (3*MAD_V))) | ((data['Delta_V']) < (median_V - (3*MAD_V)))
x['anomaly_V'] = anomaly_V
x['anomaly_V'] = np.where(x['anomaly_V'] == True, 'Outlier', '-')
# TERTIARY OPERATOR FOR DATA FRAME 
# df['Label'] = np.where(df['Delta_V'] > 10, 'High', 'Low')
# ANOMALIES IN V WHICH ARE MORE THAN 3*MAD DISTANCE AWAY FROM THEIR MEDIAN AND ARE THUS FLAGGED

median_R_int = data['Delta_R_int'].median()
MAD_R_int = (k) * (abs(data['Delta_R_int'] - median_R_int)).median()
anomaly_R_int = ((data['Delta_R_int']) > (median_R_int + (3*MAD_R_int))) | ((data['Delta_R_int']) < (median_R_int - (3*MAD_R_int)))
x['anomaly_R_int'] = anomaly_R_int
x['anomaly_R_int'] = np.where(x['anomaly_R_int'] == True, 'Outlier', '-')

median_C = data['Delta_C'].median()
MAD_C = (k) * (abs(data['Delta_C'] - median_C)).median()
anomaly_C = ((data['Delta_C']) > (median_C + (3*MAD_C))) | ((data['Delta_C']) < (median_C - (3*MAD_C)))
x['anomaly_C'] = anomaly_C
x['anomaly_C'] = np.where(x['anomaly_C'] == True, 'Outlier', '-')

median_T = data['Delta_T'].median()
MAD_T = (k) * (abs(data['Delta_T'] - median_T)).median()
anomaly_T = ((data['Delta_T']) > (median_T + (3*MAD_T))) | ((data['Delta_T']) < (median_T - (3*MAD_T)))
x['anomaly_T'] = anomaly_T
x['anomaly_T'] = np.where(x['anomaly_T'] == True, 'Outlier', '-')


print(x.to_string())

          Delta_V  Delta_R_int  Delta_C  Delta_T anomaly_V anomaly_R_int anomaly_C anomaly_T
cell_id                                                                                     
Cell_001    0.021        -0.74    0.024     -0.8         -             -         -         -
Cell_002    0.028        -0.78    0.035     -0.5         -             -         -         -
Cell_003    0.017        -0.62    0.035     -0.5         -             -         -         -
Cell_004    0.027        -1.50    0.054     -0.3         -             -         -         -
Cell_005    0.014        -0.87    0.030     -0.9         -             -         -         -
Cell_006    0.014        -0.87    0.028     -0.7         -             -         -         -
Cell_007    0.013        -0.71    0.024     -0.3         -             -         -         -
Cell_008    0.024        -0.54    0.044     -0.4         -             -         -         -
Cell_009    0.021        -1.04    0.035     -0.2         -            

#### MAHALANOBIS DISTANCE

In [157]:
import math
import numpy as np
from sklearn.covariance import MinCovDet  # MinCovDet IS THE CLASS THAT GIVE THE MAHALANOBIS DISTANCE

# IF WE SEE THAT ANY ROW HAS NaN THEN WE DROP THAT ENTIRE ROW BECAUSE WE CANT HAVE ONE VALUE MISSING
# another option was to fill missing values with their mean but that may give slightly wrong output so safer to drop it

x = data[['Delta_V', 'Delta_R_int', 'Delta_C', 'Delta_T']]
x = x.dropna()

# CELL HERE ONWARDS IS VIBECODED FOR THE FUNCTION NAMES -> too lazy to read the documentation :)

mcd = MinCovDet(random_state=42) # WEIRDEST PART ABOUT THIS CODE 
# Setting random_state=42 ensures identical results on every run; without it, the algorithm picks different random starting points, causing distance scores to slightly change each time.
# ITS LIKE A MINECRAFT SEED -> WE ARE FIXING THE SEED SO WE GET SIMILAR OUTPUT ( 42 is standard practice but the number can be anything)
mcd.fit(x)
print(mcd.location_ , '\n')       # robust center of the 4 deltas
print(mcd.covariance_, '\n')     # robust 4x4 covariance matrix -> above two are calculated internally

mahalanobis = mcd.mahalanobis(x)     # SQUARED distance, one value per cell
x['maha_dist'] = np.sqrt(mahalanobis)     # same trick as np.sqrt(mse_score) -> rmse_score

# mahalanobu distance > chi ( p, 1-alpha) 
# chi from chi squared dist. , p is no of features, alpha = 0.225 (so we are rejecting top 2.5%)
# chi^2 (0.975, p=4) = 11.143286781877796
chi = math.sqrt(11.143286781877796)

x['rank'] = x['maha_dist'].rank(ascending=False)

x['predicted_mahalanobis_outlier_status'] = np.where(x['maha_dist'] > chi, 'Outlier', '-')

ranked_data_mahalanobis = x.sort_values(by='maha_dist', ascending=False) # Sorting the maha_dist values

print(ranked_data_mahalanobis.to_string())

[ 0.01820833 -0.7305      0.03008333 -0.4825    ] 

[[ 3.72544570e-05 -5.17751758e-04  2.03039470e-05  9.36951719e-05]
 [-5.17751758e-04  5.21106600e-02 -1.15277217e-03  6.10338111e-03]
 [ 2.03039470e-05 -1.15277217e-03  7.43987714e-05 -1.32836469e-04]
 [ 9.36951719e-05  6.10338111e-03 -1.32836469e-04  1.42401138e-01]] 

          Delta_V  Delta_R_int  Delta_C  Delta_T  maha_dist   rank predicted_mahalanobis_outlier_status
cell_id                                                                                                
Cell_045    0.052       -13.53    0.138     -1.1  62.503146    1.0                              Outlier
Cell_106    0.032        -8.49    0.129     -0.9  36.742136    2.0                              Outlier
Cell_094    0.224        -2.11    0.078     -0.5  34.891771    3.0                              Outlier
Cell_085    0.049        -8.08    0.107     -1.5  34.785665    4.0                              Outlier
Cell_053    0.220        -2.25    0.103     -0.9  33.

#### USING ISOLATION FOREST METHOD

In [158]:
import numpy as np
from sklearn.ensemble import IsolationForest

x = data[['Delta_V', 'Delta_R_int', 'Delta_C', 'Delta_T']] 
x = x.dropna()

iso = IsolationForest(contamination=0.10, random_state=42)
# Conntamination tells the model that the top 10% can be flagged  as anomaly

iso.fit(x)
score = iso.score_samples(x)             # LOWER = more anomalous (sklearn's convention)
x['pred'] = iso.predict(x)                # -1 = outlier, 1 = normal

# VERY IMPORTANT
# THE  X THAT GOES INSIDE ALL THE FUNCTIONS ASSOIATED WITH THE METHOD SHOULD BE THE SAME AS THE X THAT GOES INSIDE .fit(x)

print(score, '\n')                             # we see that we get a negative output
x['isolated_forest_score'] = -score      # HIGHER = more suspicious, like maha_dist
x['predicted_isolation_forest_outlier_status'] = np.where(x['pred'] == -1, 'Outlier', '-')

x = x.drop(columns = ['pred'])
x['rank'] = x['isolated_forest_score'].rank(ascending=False)
ranked_data_isolated_forest = x.sort_values(by='isolated_forest_score', ascending=False) 
print(ranked_data_isolated_forest.to_string())

[-0.35858095 -0.36945073 -0.33931434 -0.43210316 -0.37251572 -0.35886628
 -0.36987485 -0.38490294 -0.3694194  -0.38930804 -0.38739589 -0.44304047
 -0.58234481 -0.49119519 -0.35937088 -0.56221415 -0.41568767 -0.43343133
 -0.33550744 -0.3685566  -0.42410709 -0.44297499 -0.45125622 -0.40493938
 -0.43250848 -0.36744179 -0.37842333 -0.35451693 -0.43028657 -0.37740053
 -0.33339573 -0.34206059 -0.34576595 -0.33694182 -0.38297838 -0.36263155
 -0.35494176 -0.35907586 -0.40318528 -0.40963625 -0.39307161 -0.41277971
 -0.73901365 -0.37549897 -0.38719303 -0.39332492 -0.38567817 -0.5326389
 -0.41134892 -0.36787674 -0.67457571 -0.38170739 -0.45466065 -0.41163145
 -0.36678187 -0.62383597 -0.36946947 -0.34295921 -0.37987594 -0.3958972
 -0.65555057 -0.35049962 -0.38968543 -0.42431897 -0.52651444 -0.34229577
 -0.36489087 -0.3715851  -0.44454228 -0.35810837 -0.56845731 -0.38758297
 -0.37035981 -0.39020412 -0.34442923 -0.45871857 -0.38944458 -0.35836583
 -0.44636439 -0.40519342 -0.41574638 -0.34310652 -0.6

## REGRESSION PROBLEM - PREDICT CONDITION AT CYCLE 200 USING DIFFERENT METHODS

IDEALLY WE  SHOULD CHECK IF ALL FEATURES ARE NECESSARY USING CROSS VALIDATION BUT HERE WE CAN ASSUME THAT WE CAN ASSUME THAT ALL DATA IS NECESSARY BECAUSE THIS  IS TIME STAMPED

In [159]:
# CURRENT ASSUMPTION : I AM ASSUMING ALL TEST RIGS TO BE SAME ELSE WE WOULD FALL SHORT OF DATA
# also we are predicting cycle 200 with cycle 0,5,10,25,50 - WE CAN TRY OUT FOR DIFFERENT VALUES WHICH GIVE US THE BEST PREDICTION

data = data.dropna()
x_V = data[['V_cycle0', 'V_cycle5', 'V_cycle10', 'V_cycle25']] #WE CAN CHANGE WHAT WE ARE TRYING OUT HERE

x_R_int = data[['R_int_cycle0', 'R_int_cycle5', 'R_int_cycle10', 'R_int_cycle25']]
x_C = data[['C_cycle0', 'C_cycle5', 'C_cycle10', 'C_cycle25']]
x_T = data[['T_cycle0', 'T_cycle5', 'T_cycle10', 'T_cycle25']]
y_V = data['V_cycle200']
y_R_int = data['R_int_cycle200']
y_C = data['C_cycle200']
y_T = data['T_cycle200']

##### I AM ONLY COMPARING THE DIFFERENT MODELSFOR V AS WE KNOW THE GENERAL RESULT BUT FOR VERIFICATION PURPOSE, IF WE DO FOR R_INT, C OR T WW WILL GET SIMILAR RESULT (we are going to use Elastic net in the end)

In [160]:
# imported the methods to call from Claude but I know what is going on under the hood -> reading doccumentation is a lot of work :) (read it later)  

import numpy as np
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler                          # used to scale al values TO THEIR Z SCORES SO THAT CERTAIN METHODS CAN WORK ACCURATELY -> (value - mean) / standard deviation
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV           # 'CV' is its ability to try many alpha values and use the best one
from xgboost import XGBRegressor

cv = KFold(n_splits=10, shuffle=True, random_state=2)                     # 10 folds like before, rows shuffled

# ---- Ridge ----
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50)))
ridge_mse = -cross_val_score(ridge, x_V, y_V, cv=cv, scoring="neg_mean_squared_error")
ridge_rmse = np.sqrt(ridge_mse)

# ---- Lasso ----
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=20000))
lasso_mse = -cross_val_score(lasso, x_V, y_V, cv=cv, scoring="neg_mean_squared_error")
lasso_rmse = np.sqrt(lasso_mse)

# ---- Elastic Net ----
enet = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5, max_iter=20000))
enet_mse = -cross_val_score(enet, x_V, y_V, cv=cv, scoring="neg_mean_squared_error")
enet_rmse = np.sqrt(enet_mse)

# ---- XGBoost ----
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=2)
xgb_mse = -cross_val_score(xgb, x_V, y_V, cv=cv, scoring="neg_mean_squared_error")
xgb_rmse = np.sqrt(xgb_mse)

# ---- Results (lower is better) ----
print("Ridge        RMSE:", ridge_rmse.mean())
print("Lasso        RMSE:", lasso_rmse.mean())
print("ElasticNet   RMSE:", enet_rmse.mean())
print("XGB          RMSE:", xgb_rmse.mean())

Ridge        RMSE: 0.022179184951106413
Lasso        RMSE: 0.022252611731219434
ElasticNet   RMSE: 0.022286723187331568
XGB          RMSE: 0.03831783322552977


In [161]:
# USING ELASTIC NET TO PREDICT CYCLE 200 VALUES only for V - rest has similar process :)

In [162]:
import numpy as np
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cv = KFold(n_splits=10, shuffle=True, random_state=2)

enet = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5, max_iter=20000))

# each row is predicted by a model that never saw that row
pred_V200 = cross_val_predict(enet, x_V, y_V, cv=cv)

# put actual vs predicted side by side
result = x_V.copy()
result["actual_V_cycle200"] = y_V
result["predicted_V_cycle200"] = pred_V200
result["error"] = result["predicted_V_cycle200"] - result["actual_V_cycle200"]

print(result.to_string())
print("RMSE:", np.sqrt(np.mean(result["error"] ** 2)))

          V_cycle0  V_cycle5  V_cycle10  V_cycle25  actual_V_cycle200  predicted_V_cycle200     error
cell_id                                                                                              
Cell_002     4.185     4.172      4.172      4.157              4.090              4.055117 -0.034883
Cell_003     4.218     4.207      4.208      4.201              4.129              4.149243  0.020243
Cell_004     4.186     4.185      4.175      4.159              4.061              4.049641 -0.011359
Cell_005     4.165     4.161      4.155      4.151              4.077              4.095476  0.018476
Cell_006     4.188     4.187      4.191      4.174              4.099              4.122483  0.023483
Cell_007     4.168     4.180      4.165      4.155              4.090              4.084970 -0.005030
Cell_008     4.176     4.170      4.166      4.152              4.075              4.059424 -0.015576
Cell_009     4.187     4.195      4.186      4.166              4.098             

In [163]:
# WE ARE GOING TO USE ELASTIC NET BUT FOR TESTING SOME DATA, WE ARE USING BOTH RIDGE AND ELASTIC NET
import pandas as pd
from sklearn.linear_model import RidgeCV, ElasticNetCV  
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50)))
enet = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5, max_iter=20000))
ridge.fit(x_V, y_V)
enet.fit(x_V, y_V)

V_cycle_0  = 4.198
V_cycle_5  = 4.186
V_cycle_10 = 4.170
V_cycle_25 = 4.125

new_data = pd.DataFrame([[V_cycle_0, V_cycle_5, V_cycle_10, V_cycle_25]],columns=['V_cycle0', 'V_cycle5', 'V_cycle10', 'V_cycle25'])

predicted_ridge = ridge.predict(new_data)
predicted_enet = enet.predict(new_data)
new_data["Predicted_V_cycle200_ridge"] = predicted_ridge
new_data["Predicted_V_cycle200_enet"] = predicted_enet

print(new_data)

   V_cycle0  V_cycle5  V_cycle10  V_cycle25  Predicted_V_cycle200_ridge  \
0     4.198     4.186       4.17      4.125                    3.842601   

   Predicted_V_cycle200_enet  
0                   3.843201  
